In [14]:
!nvidia-smi

Sat Nov 28 12:42:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   74C    P0             30W /   70W |     109MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [15]:
from numba import cuda, float64, int64
from numba.cuda.random import create_xoroshiro128p_states
from numba.cuda.random import xoroshiro128p_normal_float32
from numba.cuda.random import xoroshiro128p_uniform_float32

import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import pandas as pd
import numpy as np
import math
import time

In [16]:
L = 64
N = L * L

J_H = 1.0      # Kopling Hund (skala energi)
t_hop = 1.0    # Hopping t
lam_R = 0.5    # Rashba SOC
lam_D = 0.5    # Dresselhaus SOC
B_z = 0.0      # Magnetic Field

# replicas = 32
adapt_every = 500
sample_every = 1000
seed_rd = 2026

replicas = 16
n_therm = 100000
n_measure = 200000
sigma_initial = 0.6

T = np.linspace(2.0, 0.1, 20)

In [17]:
def random_spin_field():
    spins = np.random.normal(size=(L, L, 3))
    spins /= np.linalg.norm(spins, axis=2, keepdims=True)
    return spins

np.random.seed(123)
lattice = random_spin_field()

In [18]:
@cuda.jit
def gpu_sweeps_rd(spins, counts, rng_states, temp, d_sigma, B_z, Y, t_hop, lam_R, lam_D, sweeps):

    replica = cuda.blockIdx.x
    tid = cuda.threadIdx.x
    n_threads = cuda.blockDim.x

    nx = spins.shape[1]
    ny = spins.shape[2]
    half_ny = ny // 2
    half_N = nx * half_ny

    rid = replica * n_threads + tid

    sigma = d_sigma[0]
    accepted = 0

    c_heis = Y * (t_hop * t_hop - lam_R * lam_R - lam_D * lam_D)
    c_2lr2 = 2.0 * Y * lam_R * lam_R
    c_2ld2 = 2.0 * Y * lam_D * lam_D
    c_2lrlD = 2.0 * Y * lam_R * lam_D
    c_dmR = 2.0 * Y * t_hop * lam_R
    c_dmD = 2.0 * Y * t_hop * lam_D

    for sweep in range(sweeps):
        for color in range(2):  # 0 = hitam, 1 = putih
            for idx in range(tid, half_N, n_threads):

                x = idx // half_ny
                y = 2 * (idx % half_ny) + ((x + color) & 1)

                xp = (x + 1) % nx
                xm = (x - 1) % nx
                yp = (y + 1) % ny
                ym = (y - 1) % ny

                old_x = spins[replica, x, y, 0]
                old_y = spins[replica, x, y, 1]
                old_z = spins[replica, x, y, 2]

                tx = old_x + sigma * xoroshiro128p_normal_float32(rng_states, rid)
                ty = old_y + sigma * xoroshiro128p_normal_float32(rng_states, rid)
                tz = old_z + sigma * xoroshiro128p_normal_float32(rng_states, rid)

                norm = math.sqrt(tx*tx + ty*ty + tz*tz)

                if norm > 0.0:
                    tx /= norm
                    ty /= norm
                    tz /= norm

                    dx = tx - old_x
                    dy = ty - old_y
                    dz = tz - old_z

                    rx = spins[replica, xp, y, 0]; ry = spins[replica, xp, y, 1]; rz = spins[replica, xp, y, 2]
                    lx = spins[replica, xm, y, 0]; ly = spins[replica, xm, y, 1]; lz = spins[replica, xm, y, 2]
                    ux = spins[replica, x, yp, 0]; uy = spins[replica, x, yp, 1]; uz = spins[replica, x, yp, 2]
                    bx = spins[replica, x, ym, 0]; by = spins[replica, x, ym, 1]; bz = spins[replica, x, ym, 2]

                    gx = (c_heis * (rx + lx + ux + bx)
                          + c_2ld2 * (rx + lx) + c_2lr2 * (ux + bx)
                          + c_2lrlD * (ry + ly - uy - by)
                          + c_dmR * (lz - rz) + c_dmD * (bz - uz))

                    gy = (c_heis * (ry + ly + uy + by)
                          + c_2lr2 * (ry + ly) + c_2ld2 * (uy + by)
                          + c_2lrlD * (rx + lx - ux - bx)
                          + c_dmR * (bz - uz) + c_dmD * (rz - lz))

                    gz = (c_heis * (rz + lz + uz + bz)
                          + c_dmR * (rx - lx + uy - by)
                          + c_dmD * (ly - ry + ux - bx)
                          - B_z)

                    dE = dx * gx + dy * gy + dz * gz

                    if dE <= 0.0 or (temp > 0.0 and xoroshiro128p_uniform_float32(rng_states, rid) < math.exp(-dE / temp)):
                        spins[replica, x, y, 0] = tx
                        spins[replica, x, y, 1] = ty
                        spins[replica, x, y, 2] = tz
                        accepted += 1

            cuda.syncthreads()

    counts[replica, tid] = accepted

In [19]:
@cuda.jit
def measure_gpu_rd(spins, energies, magnetizations, B_z, Y, t_hop, lam_R, lam_D):

    r = cuda.blockIdx.x
    tid = cuda.threadIdx.x

    nx = spins.shape[1]
    ny = spins.shape[2]

    sh_e = cuda.shared.array(256, dtype=float64)
    sh_mx = cuda.shared.array(256, dtype=float64)
    sh_my = cuda.shared.array(256, dtype=float64)
    sh_mz = cuda.shared.array(256, dtype=float64)

    c_const = -2.0 * Y * (t_hop * t_hop + lam_R * lam_R + lam_D * lam_D)
    c_heis = Y * (t_hop * t_hop - lam_R * lam_R - lam_D * lam_D)
    c_2lr2 = 2.0 * Y * lam_R * lam_R
    c_2ld2 = 2.0 * Y * lam_D * lam_D
    c_2lrlD = 2.0 * Y * lam_R * lam_D
    c_dmR = 2.0 * Y * t_hop * lam_R
    c_dmD = 2.0 * Y * t_hop * lam_D

    e = 0.0
    mx = 0.0
    my = 0.0
    mz = 0.0

    for k in range(tid, nx * ny, cuda.blockDim.x):
        x = k // ny
        y = k % ny

        xp = (x + 1) % nx
        yp = (y + 1) % ny

        sx = float64(spins[r, x, y, 0])
        sy = float64(spins[r, x, y, 1])
        sz = float64(spins[r, x, y, 2])

        # Tetangga +x (gamma = x, gamma' = +y)
        ax = float64(spins[r, xp, y, 0])
        ay = float64(spins[r, xp, y, 1])
        az = float64(spins[r, xp, y, 2])

        # Tetangga +y (gamma = y, gamma' = -x)
        bx = float64(spins[r, x, yp, 0])
        by = float64(spins[r, x, yp, 1])
        bz = float64(spins[r, x, yp, 2])

        # Suku konstanta + Heisenberg + Zeeman
        e += c_const
        e += c_heis * (sx * (ax + bx) + sy * (ay + by) + sz * (az + bz))
        e -= B_z * sz

        # Ikatan arah +x
        e += c_2lr2 * (sy * ay) + c_2ld2 * (sx * ax)
        e += c_2lrlD * (sy * ax + sx * ay)
        e += c_dmR * (sz * ax - sx * az) + c_dmD * (sy * az - sz * ay)

        # Ikatan arah +y
        e += c_2lr2 * (sx * bx) + c_2ld2 * (sy * by)
        e -= c_2lrlD * (sx * by + sy * bx)
        e -= c_dmR * (sy * bz - sz * by)
        e += c_dmD * (sz * bx - sx * bz)

        mx += sx
        my += sy
        mz += sz

    sh_e[tid] = e
    sh_mx[tid] = mx
    sh_my[tid] = my
    sh_mz[tid] = mz

    cuda.syncthreads()

    stride = cuda.blockDim.x // 2
    while stride > 0:
        if tid < stride:
            sh_e[tid] += sh_e[tid + stride]
            sh_mx[tid] += sh_mx[tid + stride]
            sh_my[tid] += sh_my[tid + stride]
            sh_mz[tid] += sh_mz[tid + stride]
        cuda.syncthreads()
        stride //= 2

    if tid == 0:
        energies[r] = sh_e[0]
        magnetizations[r, 0] = sh_mx[0]
        magnetizations[r, 1] = sh_my[0]
        magnetizations[r, 2] = sh_mz[0]

In [20]:
@cuda.jit
def adapt_sigma_gpu(counts, d_sigma, N, sweeps):

    tid = cuda.threadIdx.x
    ncols = counts.shape[1]
    total = 0

    for i in range(tid, counts.shape[0] * ncols, 256):
        total += counts[i // ncols, i % ncols]

    sh = cuda.shared.array(256, dtype=int64)
    sh[tid] = total

    cuda.syncthreads()

    stride = 128

    while stride > 0:
        if tid < stride:
            sh[tid] += sh[tid + stride]

        cuda.syncthreads()
        stride //= 2

    if tid == 0:
        R = sh[0] / (counts.shape[0] * N * sweeps)
        sigma = d_sigma[0] * math.exp(0.5 * (R - 0.5))
        d_sigma[0] = min(max(sigma, 1e-3), 2.0)


In [21]:

@cuda.jit
def accumulate_stats_gpu(d_energies, d_magnetizations, counts, stats):

    r = cuda.grid(1)

    if r < d_energies.shape[0]:
        n = stats[r, 0] + 1.0

        E = d_energies[r]

        delta_E = E - stats[r, 1]
        E_mean = stats[r, 1] + delta_E / n

        stats[r, 2] += delta_E * (E - E_mean)
        stats[r, 1] = E_mean

        mx = d_magnetizations[r, 0]
        my = d_magnetizations[r, 1]
        mz = d_magnetizations[r, 2]

        M_abs = math.sqrt(mx*mx + my*my + mz*mz)

        delta_M = M_abs - stats[r, 3]
        M_abs_mean = stats[r, 3] + delta_M / n

        stats[r, 4] += delta_M * (M_abs - M_abs_mean)
        stats[r, 3] = M_abs_mean
        stats[r, 0] = n

        accepted = 0

        for k in range(counts.shape[1]):
            accepted += counts[r, k]

        stats[r, 5] += accepted


In [22]:
@cuda.jit
def finalize_stats_gpu(stats, d_sigma, result, temp, N, n_measure):

    if cuda.grid(1) == 0:
        n_total = 0.0

        E_mean = 0.0
        E_m2 = 0.0

        M_abs_mean = 0.0
        M_m2 = 0.0

        accepted_sum = 0.0

        for r in range(stats.shape[0]):
            n = stats[r, 0]
            new_total = n_total + n

            delta_E = stats[r, 1] - E_mean
            delta_M = stats[r, 3] - M_abs_mean

            E_m2 += (stats[r, 2] + delta_E * delta_E * n_total * n / new_total)

            M_m2 += (stats[r, 4] + delta_M * delta_M * n_total * n / new_total)

            E_mean += delta_E * n / new_total
            M_abs_mean += delta_M * n / new_total

            n_total = new_total
            accepted_sum += stats[r, 5]

        result[0] = E_m2 / (n_total * N * temp ** 2)
        result[1] = M_m2 / (n_total * N * temp)
        result[2] = M_abs_mean
        result[3] = E_mean
        result[4] = accepted_sum / (stats.shape[0] * N * n_measure)
        result[5] = d_sigma[0]

In [23]:
def simulate_temp_rd(lattice, T, B_z, J_H, t_hop, lam_R, lam_D, label,
                     replicas, adapt_every, sample_every, seed,
                     n_therm_run, n_measure_run, sigma_initial):

    nx, ny = lattice.shape[:2]
    N = nx * ny
    n_threads = 512  # 2-color checkerboard stride loop

    rng = np.random.default_rng(seed)
    batch = rng.normal(size=(replicas, nx, ny, 3)).astype(np.float32)
    batch /= np.linalg.norm(batch, axis=3, keepdims=True)
    batch[0] = lattice.astype(np.float32)

    d_spins = cuda.to_device(batch)
    d_counts = cuda.device_array((replicas, n_threads), dtype=np.int32)
    d_energies = cuda.device_array(replicas, dtype=np.float64)
    d_magnetizations = cuda.device_array((replicas, 3), dtype=np.float64)
    d_sigma = cuda.to_device(np.array([sigma_initial], dtype=np.float32))
    d_stats = cuda.device_array((replicas, 6), dtype=np.float64)
    d_result = cuda.device_array(6, dtype=np.float64)

    rng_states = create_xoroshiro128p_states(replicas * n_threads, seed=seed)

    Y_gpu = np.float32(1.0 / J_H)
    t_gpu = np.float32(t_hop)
    lr_gpu = np.float32(lam_R)
    ld_gpu = np.float32(lam_D)
    B_gpu = np.float32(B_z)

    stats_zero = np.zeros((replicas, 6), dtype=np.float64)
    sigma_start = np.array([sigma_initial], dtype=np.float32)

    c_vs_t, kai_vs_t, m_vs_t, e_vs_t, r_vs_t, sigma_vs_t = [], [], [], [], [], []
    snapshots = {}

    for t in T:
        start_time = time.perf_counter()
        temp = np.float32(t)

        d_stats.copy_to_device(stats_zero)
        d_sigma.copy_to_device(sigma_start)

        for start in range(0, n_therm_run, adapt_every):
            steps = min(adapt_every, n_therm_run - start)
            gpu_sweeps_rd[replicas, n_threads](
                d_spins, d_counts, rng_states, temp, d_sigma,
                B_gpu, Y_gpu, t_gpu, lr_gpu, ld_gpu, steps
            )
            adapt_sigma_gpu[1, 256](d_counts, d_sigma, N, steps)

        for start in range(0, n_measure_run, sample_every):
            steps = min(sample_every, n_measure_run - start)
            gpu_sweeps_rd[replicas, n_threads](
                d_spins, d_counts, rng_states, temp, d_sigma,
                B_gpu, Y_gpu, t_gpu, lr_gpu, ld_gpu, steps
            )
            measure_gpu_rd[replicas, 256](
                d_spins, d_energies, d_magnetizations,
                B_gpu, Y_gpu, t_gpu, lr_gpu, ld_gpu
            )
            accumulate_stats_gpu[(replicas + 127) // 128, 128](
                d_energies, d_magnetizations, d_counts, d_stats
            )

        finalize_stats_gpu[1, 1](d_stats, d_sigma, d_result, temp, N, n_measure_run)
        c, Kai, M_abs_mean, E_mean, R_mean, sigma = d_result.copy_to_host()

        c_vs_t.append(c); kai_vs_t.append(Kai)
        m_vs_t.append(M_abs_mean / N); e_vs_t.append(E_mean / N)
        r_vs_t.append(R_mean); sigma_vs_t.append(sigma)

        elapsed = time.perf_counter() - start_time
        print(f"{label} | T={t:.4f} | Time={elapsed:.2f}s | E/N={E_mean/N:.4f} | "
              f"M/N={M_abs_mean/N:.4f} | C={c:.4f} | Chi={Kai:.4f} | R={R_mean:.4f}", flush=True)

        if np.any(np.isclose(t, T_plot)):
            snapshots[float(np.round(t, 4))] = d_spins.copy_to_host()

    return (np.array(c_vs_t), np.array(kai_vs_t), np.array(m_vs_t),
            np.array(e_vs_t), np.array(r_vs_t), np.array(sigma_vs_t), snapshots)

In [24]:
def static_structure_factor(spins):
    replicas, nx, ny, _ = spins.shape
    F = np.fft.fftshift(np.fft.fft2(spins, axes=(1, 2)), axes=(1, 2))
    S_q = np.mean(np.sum(np.abs(F) ** 2, axis=-1), axis=0) / (nx * ny) ** 2
    qx = 2 * np.pi * np.fft.fftshift(np.fft.fftfreq(nx))
    qy = 2 * np.pi * np.fft.fftshift(np.fft.fftfreq(ny))
    return qx, qy, S_q

In [ ]:
T_plot = np.array([0.1, 0.5, 1.0, 2.0])

c_rd, kai_rd, m_rd, e_rd, r_rd, sig_rd, spins_rd = simulate_temp_rd(
    lattice, T, B_z, J_H, t_hop, lam_R, lam_D, "rd_05_05",
    replicas, adapt_every, sample_every, seed_rd,
    n_therm, n_measure, sigma_initial
)

# Plot Cv, Chi, dan S(q) pada T=0.1
fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)

axes[0].plot(T, c_rd, '-o', color='#FF0202', markersize=4)
axes[0].set_xlabel(r'$T$')
axes[0].set_ylabel(r'$C_v$')
axes[0].set_title('Heat Capacity')
axes[0].grid(alpha=0.25)

axes[1].plot(T, kai_rd, '-s', color='#0055CC', markersize=4)
axes[1].set_xlabel(r'$T$')
axes[1].set_ylabel(r'$\chi$')
axes[1].set_title('Susceptibility')
axes[1].grid(alpha=0.25)

qx, qy, Sq = static_structure_factor(spins_rd[0.1])
im = axes[2].pcolormesh(qx / np.pi, qy / np.pi, Sq.T, cmap='inferno', shading='auto')
axes[2].set_xlabel(r'$q_x/\pi$')
axes[2].set_ylabel(r'$q_y/\pi$')
axes[2].set_title(r'$S(\mathbf{q})/N$ at $T=0.1$')
axes[2].set_aspect('equal')
fig.colorbar(im, ax=axes[2])

plt.show()

/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 16 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 16 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: Numba

rd_05_05 | T=2.0000 | Time=39.03s | E/N=-3.7565 | M/N=0.0105 | C=0.3758 | Chi=0.0424 | R=0.6289
